# Analyse de portefeuille GAFAM par Shrinkage (Ledoit-Wolf)

## Objectif
Construire et évaluer un portefeuille **Minimum Variance** sur les actions GAFAM (`AAPL`, `AMZN`, `GOOGL`, `META`, `MSFT`) en comparant sa performance au Nasdaq 100 (`^NDX`).

## Plan de l'étude
1. Chargement et préparation des données (Adj Close)
2. Passage en rendements logarithmiques
3. Estimation de covariance par **Ledoit-Wolf Shrinkage**
4. Construction des poids minimum variance en fenêtre glissante
5. Backtest Out-of-Sample (2025-2026)
6. Mesures de performance : rendement, volatilité, Sharpe, max drawdown, alpha de Jensen

In [ ]:
# =========================
# 1) Imports et paramètres
# =========================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.covariance import LedoitWolf

plt.style.use("seaborn-v0_8")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Univers d'investissement
ASSETS = ["AAPL", "AMZN", "GOOGL", "META", "MSFT"]
BENCHMARK = "^NDX"

# Périodes
aaa_start = "2015-01-01"
in_sample_end = "2024-12-31"
oos_start = "2025-01-01"
oos_end = "2026-12-31"

ROLLING_WINDOW = 60
TRADING_DAYS = 252

## Pourquoi les rendements logarithmiques ? (Stationnarité)

Les prix financiers sont généralement non stationnaires : leur moyenne et leur variance évoluent dans le temps.

Pour limiter ce problème, on travaille avec les **rendements logarithmiques** :
\[
 r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
\]

Ces rendements sont souvent plus proches d'un processus stationnaire que les prix, ce qui est préférable pour l'estimation de covariance et l'optimisation de portefeuille.

In [ ]:
# ==========================================
# 2) Téléchargement et préparation des données
# ==========================================

all_tickers = ASSETS + [BENCHMARK]

raw = yf.download(
    all_tickers,
    start=aaa_start,
    end=oos_end,
    auto_adjust=False,
    progress=False,
)["Adj Close"].dropna(how="all")

# Séparation actifs / benchmark
prices_assets = raw[ASSETS].dropna()
prices_bench = raw[BENCHMARK].dropna()

# Rendements logarithmiques
logret_assets = np.log(prices_assets / prices_assets.shift(1)).dropna()
logret_bench = np.log(prices_bench / prices_bench.shift(1)).dropna()

# Alignement exact des dates
common_idx = logret_assets.index.intersection(logret_bench.index)
logret_assets = logret_assets.loc[common_idx]
logret_bench = logret_bench.loc[common_idx]

print("Période disponible:", common_idx.min().date(), "->", common_idx.max().date())
print("Nombre d'observations actifs:", len(logret_assets))
print("Nombre d'observations benchmark:", len(logret_bench))

## Covariance historique vs covariance Shrinkage

La matrice de covariance historique (échantillonnale) peut être instable, surtout avec des fenêtres courtes.

**Ledoit-Wolf Shrinkage** réduit ce bruit en combinant :
- la covariance empirique,
- une cible structurée plus stable.

Le résultat est une estimation mieux conditionnée, utile pour l'optimisation minimum variance.

In [ ]:
# =============================================
# 3) Fonctions utilitaires (modulaires et claires)
# =============================================

def min_variance_weights(cov_matrix: np.ndarray) -> np.ndarray:
    """Calcule les poids du portefeuille minimum variance avec contrainte somme(w)=1."""
    n = cov_matrix.shape[0]
    ones = np.ones(n)
    inv_cov = np.linalg.pinv(cov_matrix)
    w = inv_cov @ ones
    w /= (ones.T @ inv_cov @ ones)
    return w


def compute_max_drawdown(wealth: pd.Series) -> float:
    """Calcule le drawdown maximum d'une courbe de richesse."""
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0
    return drawdown.min()


def jensen_alpha(port_ret: pd.Series, bench_ret: pd.Series, trading_days: int = 252) -> float:
    """Alpha de Jensen annualisé (rf=0 par défaut)."""
    aligned = pd.concat([port_ret, bench_ret], axis=1).dropna()
    aligned.columns = ["rp", "rb"]

    cov_pb = np.cov(aligned["rp"], aligned["rb"], ddof=1)[0, 1]
    var_b = np.var(aligned["rb"], ddof=1)
    beta = cov_pb / var_b if var_b > 0 else np.nan

    alpha_daily = aligned["rp"].mean() - beta * aligned["rb"].mean()
    return alpha_daily * trading_days

## Backtest Out-of-Sample (fenêtre glissante 60 jours)

Principe : pour chaque jour de 2025-2026,
1. on prend les **60 derniers jours passés** de rendements GAFAM,
2. on estime la covariance avec **LedoitWolf**,
3. on calcule les poids minimum variance,
4. on applique ces poids au rendement du jour courant (out-of-sample).

Cela évite le biais d'anticipation (look-ahead bias).

In [ ]:
# =============================================
# 4) Rolling Ledoit-Wolf + portefeuille Min Variance
# =============================================

# Sous-échantillons in-sample / out-of-sample
in_sample_mask = (logret_assets.index >= pd.to_datetime(aaa_start)) & (logret_assets.index <= pd.to_datetime(in_sample_end))
oos_mask = (logret_assets.index >= pd.to_datetime(oos_start)) & (logret_assets.index <= pd.to_datetime(oos_end))

ret_assets_oos = logret_assets.loc[oos_mask].copy()
ret_bench_oos = logret_bench.loc[oos_mask].copy()

weights_history = []
port_returns = []
port_dates = []

for current_date in ret_assets_oos.index:
    pos = logret_assets.index.get_loc(current_date)

    # Fenêtre d'apprentissage = 60 jours AVANT la date de trading
    if pos < ROLLING_WINDOW:
        continue

    train_window = logret_assets.iloc[pos - ROLLING_WINDOW : pos]

    # Estimation Ledoit-Wolf
    lw = LedoitWolf()
    lw.fit(train_window.values)
    cov_lw = lw.covariance_

    # Poids min variance
    w = min_variance_weights(cov_lw)

    # Rendement portefeuille du jour courant (out-of-sample)
    r_t = ret_assets_oos.loc[current_date].values @ w

    weights_history.append(w)
    port_returns.append(r_t)
    port_dates.append(current_date)

# Séries finales de backtest
port_returns = pd.Series(port_returns, index=pd.DatetimeIndex(port_dates), name="Portefeuille")
bench_returns = ret_bench_oos.reindex(port_returns.index).rename("Nasdaq")
weights_df = pd.DataFrame(weights_history, index=port_returns.index, columns=ASSETS)

print("Début backtest réel:", port_returns.index.min().date())
print("Fin backtest:", port_returns.index.max().date())
print("Nombre de jours simulés:", len(port_returns))
weights_df.tail()

## Interprétation des métriques

- **Rendement annualisé** : performance moyenne annualisée.
- **Volatilité annualisée** : risque total annualisé.
- **Ratio de Sharpe** : rendement excédentaire par unité de risque (ici rf=0).
- **Max Drawdown** : perte maximale depuis un plus-haut historique.
- **Alpha de Jensen (vs Nasdaq)** : performance non expliquée par l'exposition au benchmark.

Un alpha positif suggère une surperformance ajustée du risque de marché.

In [ ]:
# =============================================
# 5) Courbe de richesse et métriques de performance
# =============================================

# Richesse base 100
wealth_port = 100 * np.exp(port_returns.cumsum())
wealth_bench = 100 * np.exp(bench_returns.cumsum())

plt.figure(figsize=(12, 6))
plt.plot(wealth_port.index, wealth_port, label="Portefeuille MinVar (Ledoit-Wolf)", linewidth=2)
plt.plot(wealth_bench.index, wealth_bench, label="Nasdaq (^NDX)", linewidth=2, alpha=0.9)
plt.title("Évolution de la richesse (base 100) - Out-of-Sample")
plt.xlabel("Date")
plt.ylabel("Richesse")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


def perf_metrics(ret: pd.Series, bench: pd.Series) -> pd.Series:
    ann_return = ret.mean() * TRADING_DAYS
    ann_vol = ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    wealth = 100 * np.exp(ret.cumsum())
    mdd = compute_max_drawdown(wealth)
    alpha = jensen_alpha(ret, bench, TRADING_DAYS)

    return pd.Series(
        {
            "Rendement annuel": ann_return,
            "Volatilité annuelle": ann_vol,
            "Sharpe": sharpe,
            "Max Drawdown": mdd,
            "Alpha de Jensen": alpha,
        }
    )

metrics_port = perf_metrics(port_returns, bench_returns)
metrics_bench = perf_metrics(bench_returns, bench_returns)

metrics_table = pd.concat(
    [metrics_port.rename("Portefeuille"), metrics_bench.rename("Nasdaq")], axis=1
)

# Format plus lisible
metrics_display = metrics_table.copy()
for col in metrics_display.columns:
    metrics_display.loc[["Rendement annuel", "Volatilité annuelle", "Max Drawdown", "Alpha de Jensen"], col] = (
        metrics_display.loc[["Rendement annuel", "Volatilité annuelle", "Max Drawdown", "Alpha de Jensen"], col] * 100
    )

print("Métriques Out-of-Sample (2025-2026)")
display(metrics_display.style.format({
    "Portefeuille": "{:.2f}",
    "Nasdaq": "{:.2f}",
}))

print("\nNote: valeurs en % sauf Sharpe (sans unité).")